# Simulador de fermentador aerobio · BT2026

**Actividad IA · Análisis de Fenómenos de Transporte**
Escuela de Ingeniería y Ciencias · Tecnológico de Monterrey

---

Este notebook contiene una app web (construida con Gradio) que permite:

1. **Determinar k<sub>L</sub>·a** por el método dinámico (gassing-out) a partir del dataset experimental asignado a tu equipo.
2. **Simular una fermentación aerobia completa** con dinámicas acopladas de biomasa/sustrato/oxígeno disuelto.
3. **Hacer ingeniería inversa del propio simulador** — la app hace supuestos que no declara en su interfaz. Tu trabajo, apoyado en la IA, es encontrarlos.

## Cómo correr esto en Colab

1. `Runtime → Run all` (Ctrl+F9). Se instalará Gradio y se lanzará la app.
2. Espera ~30 segundos. Al final de la última celda aparecerá un URL público (algo como `https://xxxx.gradio.live`).
3. Da click en ese URL. La app se abre en una pestaña nueva.
4. En el Tab 1, sube tu dataset (archivo `.csv` con dos columnas: `tiempo (s)` y `OD (mg/L)`).
5. Lee el Tab 3 para entender la tarea de auditoría.

## Si el URL no aparece

Ejecuta primero solo la celda del `pip install`, después `Runtime → Restart session`, y después corre todas de nuevo. Eso resuelve el 90% de los casos.


In [ ]:
# Colab ya trae Gradio preinstalado; esta línea solo lo actualiza si hace falta.
!pip install -q --upgrade gradio 2>&1 | tail -1

In [ ]:
"""
Simulador de fermentador aerobio
BT2026 · Análisis de Fenómenos de Transporte
"""

import numpy as np
import pandas as pd
import gradio as gr
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# --------------------------------------------------------------
# Constantes del sistema
# --------------------------------------------------------------
C_STAR = 7.5  # mg/L


# ==============================================================
# Módulo 1 — Determinación de kLa por método dinámico
# ==============================================================
def modelo_dinamico(t, kLa, C0):
    return C_STAR - (C_STAR - C0) * np.exp(-kLa * t)


def ajustar_kLa(csv_file):
    if csv_file is None:
        return None, "Sube un CSV con dos columnas: tiempo (s) y OD (mg/L)."

    df = pd.read_csv(csv_file.name)
    t = df.iloc[:, 0].values.astype(float)
    C = df.iloc[:, 1].values.astype(float)
    C0 = C[0]

    def _fit(t, kLa):
        return modelo_dinamico(t, kLa, C0)

    try:
        popt, pcov = curve_fit(_fit, t, C, p0=[0.01], bounds=(0, 10))
        kLa_s = popt[0]
        kLa_h = kLa_s * 3600
        stderr_s = np.sqrt(np.diag(pcov))[0]
        stderr_h = stderr_s * 3600

        residuales = C - _fit(t, kLa_s)
        r_std = np.std(residuales)
        ss_res = np.sum(residuales ** 2)
        ss_tot = np.sum((C - np.mean(C)) ** 2)
        r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8), sharex=True,
                                        gridspec_kw={"height_ratios": [3, 1]})

        ax1.scatter(t, C, label="Datos experimentales", color="#1a63a9", s=40, zorder=3)
        t_fit = np.linspace(t.min(), t.max(), 300)
        ax1.plot(t_fit, _fit(t_fit, kLa_s), "r-",
                 label=f"Ajuste: kLa = {kLa_h:.1f} 1/h", linewidth=2)
        ax1.axhline(C_STAR, color="gray", linestyle="--",
                    label=f"C* = {C_STAR} mg/L", alpha=0.7)
        ax1.set_ylabel("Oxígeno disuelto (mg/L)")
        ax1.set_title("Ajuste del método dinámico")
        ax1.legend()
        ax1.grid(alpha=0.3)

        ax2.scatter(t, residuales, color="#c0392b", s=25)
        ax2.axhline(0, color="black", linestyle="-", linewidth=0.8)
        ax2.set_ylabel("Residuales (mg/L)")
        ax2.set_xlabel("Tiempo (s)")
        ax2.grid(alpha=0.3)

        plt.tight_layout()

        reporte = f"""RESULTADO DEL AJUSTE
====================
kLa ajustado : {kLa_s:.5f} 1/s  =  {kLa_h:.2f} 1/h
Error estándar: {stderr_h:.2f} 1/h
C0 detectada  : {C0:.3f} mg/L
C* asumida    : {C_STAR} mg/L
R²            : {r_squared:.4f}
Std residuales: {r_std:.4f} mg/L
Puntos usados : {len(t)}

INTERPRETA CON CRITERIO:
- ¿El R² es alto y los residuales se ven aleatorios (sin patrón)?
- ¿O los residuales muestran estructura sistemática (curvatura, sesgo)?
- ¿El kLa está en el rango esperado para tu sistema físico?
- ¿Los supuestos del modelo (ver Tab 3) aplican a tu corrida real?
"""
        return fig, reporte

    except Exception as e:
        return None, f"Error en el ajuste: {e}"


# ==============================================================
# Módulo 2 — Simulación de fermentación aerobia
# ==============================================================
def modelo_fermentador(t, y, mu_max, Ks, KO2, Yxs, qO2_max, kLa):
    X, S, C = y
    S = max(S, 0.0)
    C = max(C, 0.0)

    mu = mu_max * (S / (Ks + S)) * (C / (KO2 + C))
    qO2 = qO2_max * (C / (KO2 + C))

    dXdt = mu * X
    dSdt = -(1.0 / Yxs) * mu * X
    dCdt = kLa * (C_STAR - C) - qO2 * X

    return [dXdt, dSdt, dCdt]


def simular_fermentador(mu_max_h, Ks, KO2, Yxs, qO2_max_h, kLa_h,
                        X0, S0, C0, t_final_h):
    mu_max_s = mu_max_h / 3600
    qO2_max_s = qO2_max_h / 3600
    kLa_s = kLa_h / 3600
    t_final_s = t_final_h * 3600

    sol = solve_ivp(
        modelo_fermentador,
        [0, t_final_s],
        [X0, S0, C0],
        args=(mu_max_s, Ks, KO2, Yxs, qO2_max_s, kLa_s),
        t_eval=np.linspace(0, t_final_s, 500),
        method="LSODA",
        rtol=1e-6, atol=1e-9,
    )

    if not sol.success:
        return None, f"El integrador falló: {sol.message}"

    t_h = sol.t / 3600
    X, S, C = sol.y

    fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)

    axes[0].plot(t_h, X, "-", color="#1a63a9", linewidth=2)
    axes[0].set_ylabel("Biomasa X (g/L)")
    axes[0].set_title("Simulación de fermentación aerobia")
    axes[0].grid(alpha=0.3)

    axes[1].plot(t_h, S, "-", color="#27ae60", linewidth=2)
    axes[1].set_ylabel("Sustrato S (g/L)")
    axes[1].grid(alpha=0.3)

    axes[2].plot(t_h, C, "-", color="#c0392b", linewidth=2, label="OD (mg/L)")
    axes[2].axhline(C_STAR, color="gray", linestyle="--",
                    label=f"C* = {C_STAR}", alpha=0.7)
    axes[2].axhline(0.5, color="orange", linestyle=":",
                    label="Umbral limitante (0.5 mg/L)", alpha=0.7)
    axes[2].set_ylabel("Oxígeno disuelto (mg/L)")
    axes[2].set_xlabel("Tiempo (h)")
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()

    C_min = C.min()
    t_C_min = t_h[C.argmin()]
    X_max = X.max()
    t_X_max = t_h[X.argmax()]
    limitante = "SI (O2 limitante durante la corrida)" if C_min < 0.5 else "NO"

    reporte = f"""RESULTADO DE LA SIMULACION
=========================
Duración simulada : {t_final_h:.1f} h
Xmax              : {X_max:.3f} g/L (a t = {t_X_max:.2f} h)
S final           : {S[-1]:.3f} g/L
C mínima          : {C_min:.3f} mg/L (a t = {t_C_min:.2f} h)
¿O2 fue limitante?: {limitante}

Parámetros usados:
  mu_max  = {mu_max_h} 1/h
  Ks      = {Ks} g/L
  KO2     = {KO2} mg/L
  Yxs     = {Yxs} g biomasa / g sustrato
  qO2_max = {qO2_max_h} mg O2 / (g biomasa · h)
  kLa     = {kLa_h} 1/h
  C*      = {C_STAR} mg/L

REFLEXIONA:
- ¿Los perfiles tienen sentido físico?
- ¿En qué momento el sistema se vuelve limitado por transferencia de oxígeno?
- Si aumentas kLa 3x, ¿cómo cambia la curva de biomasa? ¿Por qué?
- ¿El modelo asume algo que NO es cierto para tu fermentación real?
"""
    return fig, reporte


# ==============================================================
# Interfaz Gradio
# ==============================================================
CSS_CUSTOM = """
.gradio-container { max-width: 1100px !important; margin: auto; }
"""

with gr.Blocks(title="BT2026 · Simulador de fermentador", css=CSS_CUSTOM) as demo:
    gr.Markdown("""
    # Simulador de fermentador aerobio · BT2026
    Herramienta didáctica: determinar k_L·a por método dinámico y simular la
    fermentación completa con oxígeno disuelto.

    **No solo la uses — audítala.** Ve al Tab "Ingeniería inversa" para las instrucciones.
    """)

    with gr.Tab("① Determinación de k_L·a"):
        gr.Markdown("Sube el CSV de tu dataset (columnas: `tiempo (s)`, `C_OD (mg/L)`).")
        with gr.Row():
            with gr.Column(scale=1):
                csv_input = gr.File(label="Archivo CSV", file_types=[".csv"])
                btn_ajustar = gr.Button("Ajustar k_L·a", variant="primary", size="lg")
            with gr.Column(scale=1):
                reporte_kLa = gr.Textbox(label="Reporte", lines=14)
        plot_kLa = gr.Plot(label="Ajuste y residuales")
        btn_ajustar.click(ajustar_kLa, inputs=csv_input, outputs=[plot_kLa, reporte_kLa])

    with gr.Tab("② Simulación de fermentación"):
        gr.Markdown("""Ajusta los parámetros y corre la simulación completa. El k_L·a que
        determinaste en el Tab 1 se ingresa manualmente aquí abajo.""")
        with gr.Row():
            with gr.Column():
                gr.Markdown("**Parámetros cinéticos**")
                mu_max_in = gr.Slider(0.05, 1.0, value=0.35, step=0.01, label="μ_max (1/h)")
                Ks_in = gr.Slider(0.01, 5.0, value=0.5, step=0.01, label="Ks (g/L)")
                KO2_in = gr.Slider(0.001, 1.0, value=0.05, step=0.001,
                                   label="K_O2 (mg/L)")
                Yxs_in = gr.Slider(0.1, 0.8, value=0.4, step=0.01,
                                   label="Y_xs (g biomasa/g sustrato)")
                qO2_max_in = gr.Slider(50, 500, value=200, step=10,
                                       label="q_O2,max (mg O2 / g biomasa·h)")
            with gr.Column():
                gr.Markdown("**Transferencia y condiciones iniciales**")
                kLa_in = gr.Slider(1, 500, value=60, step=1, label="k_L·a (1/h)")
                X0_in = gr.Slider(0.01, 5.0, value=0.1, step=0.01, label="X0 (g/L)")
                S0_in = gr.Slider(1, 50, value=20, step=1, label="S0 (g/L)")
                C0_in = gr.Slider(0, 7.5, value=7.5, step=0.1, label="C0 (mg/L)")
                t_final_in = gr.Slider(1, 48, value=12, step=1, label="Tiempo final (h)")
        btn_sim = gr.Button("Simular fermentación", variant="primary", size="lg")
        plot_sim = gr.Plot(label="Perfiles de X, S, C")
        reporte_sim = gr.Textbox(label="Reporte de la simulación", lines=18)
        btn_sim.click(
            simular_fermentador,
            inputs=[mu_max_in, Ks_in, KO2_in, Yxs_in, qO2_max_in, kLa_in,
                    X0_in, S0_in, C0_in, t_final_in],
            outputs=[plot_sim, reporte_sim],
        )

    with gr.Tab("③ Ingeniería inversa"):
        gr.Markdown("""
        ## Tu tarea de auditoría

        Este simulador **hace suposiciones que no declara en la interfaz**. Tu
        trabajo, junto con tu equipo y apoyado en la IA, es encontrarlas.

        ### Preguntas guía

        **Sobre el Tab 1 (kLa):**
        1. ¿Cuál es la ecuación fundamental de la que parte el ajuste?
        2. ¿Qué asume esa ecuación sobre el sensor de OD?
        3. ¿Qué asume sobre el consumo de oxígeno durante la corrida?
        4. ¿Qué asume sobre C*? ¿De dónde salió el valor 7.5 mg/L?

        **Sobre el Tab 2 (simulación completa):**
        5. ¿Qué modelo cinético usa para el crecimiento? ¿Qué asume ese modelo?
        6. ¿Hay término de mantenimiento en el balance de sustrato? ¿Debería haberlo?
        7. ¿El k_L·a se mantiene constante durante toda la simulación? ¿Es esto realista?
        8. ¿Qué pasa si tu fermentador NO opera en agua a 30°C, 1 atm?

        ### Reto de auditoría por dataset

        Cada equipo tiene un dataset con una característica particular. Identifica
        qué **supuesto específico del simulador** hace que su modelo NO aplique
        directamente a tu dataset. Documenta el supuesto violado, cómo lo detectaste
        y qué modificación al modelo lo corregiría.

        ### Uso de IA en esta actividad

        Estás ejerciendo dos de los cinco Momentos IA del curso:

        - **M2 (IA como colega técnico):** te ayuda a leer código que quizá no dominas
          en detalle. Pregúntale que te explique la función `modelo_fermentador`
          línea por línea, o que compare Monod puro vs. Monod con inhibición.
        - **M3 (IA como sistema a auditar):** pregúntale por los supuestos que hace
          este código. Contrasta la respuesta con lo que TÚ ves leyendo el código
          directamente. Consulta al menos dos IAs distintas y compara sus respuestas.

        ### Entregable

        Un anexo al reporte del equipo con:
        - Todos los supuestos del simulador que identificaron (con evidencia en el código).
        - El supuesto violado por su dataset asignado y su corrección propuesta.
        - Registro de la interacción con las dos IAs consultadas (prompts y respuestas)
          con nota sobre discrepancias encontradas.
        """)


# ==============================================================
# Lanzar la app
# ==============================================================
demo.launch(share=True, debug=False)


---

## Nota final

Cuando termines de trabajar con la app, **detén el kernel** (menú Runtime → Interrupt execution) para liberar el URL público de Gradio. El URL `.gradio.live` es temporal y expira automáticamente en 72 horas.

Si quieres compartir la app con la clase, comparte el URL `.gradio.live` que apareció al fondo de la celda de código — pero asegúrate de que el notebook siga abierto en tu Colab o el URL se muere.

---

*Simulador diseñado como parte del curso semilla BT2026. Los datos experimentales son sintéticos pero disciplinarmente plausibles. Los supuestos hardcodeados del simulador son deliberados y forman parte del ejercicio de auditoría de estudiantes.*
